In [69]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split,cross_val_score, KFold ,GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor


In [70]:
#Load the dataset
df = pd.read_csv('D:\\IUdatapeak_datathon2026_round1\\data\\raw\\sales.csv')
df['Date'] = pd.to_datetime(df['Date'])
df

,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79
...,...,...,...
3828,2022-12-27,2100553.66,2184872.24
3829,2022-12-28,3448729.20,3513621.00
3830,2022-12-29,3083944.33,3170787.10
3831,2022-12-30,2884668.76,3022292.15


In [71]:
promo = pd.read_csv('D:\\IUdatapeak_datathon2026_round1\\data\\raw\\promotions.csv')
promo['start_date'] = pd.to_datetime(promo['start_date'])
promo['end_date'] = pd.to_datetime(promo['end_date'])
promo

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,NaN,email,1,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,NaN,online,0,0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,NaN,email,0,0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,NaN,all_channels,0,50000
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000
5,PROMO-0006,Rural Special 2013,percentage,15.0,2013-01-31,2013-03-01,Outdoor,in_store,0,0
6,PROMO-0007,Spring Sale 2014,percentage,12.0,2014-03-18,2014-04-17,NaN,email,1,0
7,PROMO-0008,Mid-Year Sale 2014,percentage,18.0,2014-06-23,2014-07-22,NaN,social_media,0,0
8,PROMO-0009,Fall Launch 2014,percentage,10.0,2014-08-30,2014-10-01,NaN,all_channels,0,100000
9,PROMO-0010,Year-End Sale 2014,percentage,20.0,2014-11-19,2015-01-02,NaN,all_channels,0,100000


In [72]:
import pandas as pd
import numpy as np

start_date = '2012-07-04'
end_date = '2022-12-31'

def create_promotion_features(promo, start_date, end_date):

    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    daily = pd.DataFrame({'Date': date_range})
    
    active_promos = []
    for idx, row in promo.iterrows():
        start = pd.to_datetime(row['start_date'])
        end = pd.to_datetime(row['end_date'])
        
        mask = (daily['Date'] >= start) & (daily['Date'] <= end)
        active_dates = daily.loc[mask, 'Date']
        
        for d in active_dates:
            active_promos.append({
                'Date': d,
                'promo_id': row['promo_id'],
                'promo_type': row['promo_type'],
                'discount_value': row['discount_value'],
                'stackable_flag': row['stackable_flag'],
                'applicable_category': row['applicable_category'],
                'promo_channel': row['promo_channel'],
                'min_order_value': row['min_order_value']
            })
    
    if not active_promos:
        daily = daily.set_index('Date')
        daily['num_active_promos'] = 0
        daily['num_promo_percentage'] = 0
        daily['num_promo_fixed'] = 0
        daily['avg_discount_value'] = 0
        daily['has_stackable'] = 0
        daily['num_specific_category_promos'] = 0
        daily = daily.reset_index()
        return daily
    
    active_df = pd.DataFrame(active_promos)
    num_active = active_df.groupby('Date')['promo_id'].nunique().rename('num_active_promos')
    
    promo_type_counts = active_df.groupby(['Date', 'promo_type']).size().unstack(fill_value=0)
    for col in ['percentage', 'fixed']:
        if col not in promo_type_counts.columns:
            promo_type_counts[col] = 0
    promo_type_counts.columns = ['num_promo_' + str(c) for c in promo_type_counts.columns]
    
    avg_discount = active_df.groupby('Date')['discount_value'].mean().rename('avg_discount_value')

    has_stackable = active_df.groupby('Date')['stackable_flag'].max().rename('has_stackable')

    specific_cat = active_df.groupby('Date')['applicable_category'].apply(
        lambda x: x.notna().sum()
    ).rename('num_specific_category_promos')

    min_order_stats = active_df.groupby('Date')['min_order_value'].agg(['min', 'max', 'mean'])
    min_order_stats.columns = ['min_order_value_min', 'min_order_value_max', 'min_order_value_mean']

    channel_dummies = pd.get_dummies(active_df[['Date', 'promo_channel']], columns=['promo_channel'], prefix='channel')

    channel_cols = [c for c in channel_dummies.columns if c.startswith('channel_')]
    channel_daily = channel_dummies.groupby('Date')[channel_cols].sum()

    daily = daily.set_index('Date')
    daily = daily.join(num_active, how='left')
    daily = daily.join(promo_type_counts, how='left')
    daily = daily.join(avg_discount, how='left')
    daily = daily.join(has_stackable, how='left')
    daily = daily.join(specific_cat, how='left')
    daily = daily.join(channel_daily, how='left')
    daily = daily.join(min_order_stats, how='left')

    daily = daily.fillna(0)
    daily = daily.reset_index()
    
    return daily
daily = create_promotion_features(promo, start_date, end_date)
daily

,Date,num_active_promos,num_promo_fixed,num_promo_percentage,avg_discount_value,has_stackable,num_specific_category_promos,channel_all_channels,channel_email,channel_in_store,channel_online,channel_social_media,min_order_value_min,min_order_value_max,min_order_value_mean
0,2012-07-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2012-07-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2012-07-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2012-07-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2012-07-08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3828,2022-12-27,1.0,0.0,1.0,20.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,50000.0,50000.0,50000.0
3829,2022-12-28,1.0,0.0,1.0,20.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,50000.0,50000.0,50000.0
3830,2022-12-29,1.0,0.0,1.0,20.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,50000.0,50000.0,50000.0
3831,2022-12-30,1.0,0.0,1.0,20.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,50000.0,50000.0,50000.0


In [73]:

def generate_future_promos(start_year=2023, end_year=2024):
    future_promos = []
    patterns = [
        ('Spring Sale', 'percentage', 12, 3, 18, 4, 17, None, 'all_channels', 1, 50000),
        ('Mid-Year Sale', 'percentage', 18, 6, 23, 7, 22, None, 'online', 0, 0),
        ('Fall Launch', 'percentage', 10, 8, 30, 10, 1, None, 'all_channels', 0, 100000),
        ('Year-End Sale', 'percentage', 20, 11, 18, 1, 2, None, 'all_channels', 0, 50000),
        ('Urban Blowout', 'fixed', 50, 7, 30, 9, 2, 'Streetwear', 'online', 0, 150000),
        ('Rural Special', 'percentage', 15, 1, 30, 3, 1, 'Outdoor', 'in_store', 0, 0),
    ]
    for year in range(start_year, end_year + 1):
        for i, (name, ptype, discount, sm, sd, em, ed, cat, channel, stackable, min_order) in enumerate(patterns):
            end_year_adj = year + 1 if (em == 1 and ed == 2) else year
            future_promos.append({
                'promo_id': f'FUTURE-{year}-{i+1:03d}',
                'promo_name': f'{name} {year}',
                'promo_type': ptype,
                'discount_value': discount,
                'start_date': pd.Timestamp(year=year, month=sm, day=sd),
                'end_date': pd.Timestamp(year=end_year_adj, month=em, day=ed),
                'applicable_category': cat,
                'promo_channel': channel,
                'stackable_flag': stackable,
                'min_order_value': min_order
            })
    return pd.DataFrame(future_promos)

future_promo = generate_future_promos(2023, 2024)
promo_all = pd.concat([promo, future_promo], ignore_index=True)

all_promo_features = create_promotion_features(promo_all, '2012-07-04', '2024-07-01')
print(f"Promo features shape: {all_promo_features.shape}")

Promo features shape: (4381, 15)


In [74]:
data['year'] = data['Date'].dt.year
data['month'] = data['Date'].dt.month
data['day'] = data['Date'].dt.day
data['dayofweek'] = data['Date'].dt.dayofweek
data['is_weekend'] = data['dayofweek'].isin([5, 6]).astype(int)
holidays_and_sales = ['01-01', '02-14', '03-08', '04-30', '05-01', '09-02', '10-20', '11-11', '12-12', '12-24', '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19', '2016-02-08', '2017-01-28', '2018-02-18', '2019-02-05', '2020-01-25', '2021-02-12', '2022-02-01', '2023-01-21']
data['is_holiday'] = data['Date'].dt.strftime('%m-%d').isin(holidays_and_sales).astype(int)
data['dayofyear'] = data['Date'].dt.dayofyear
data['dayofyearsin'] = np.sin(2 * np.pi * data['dayofyear'] / 365.25)
data['dayofyearcos'] = np.cos(2 * np.pi * data['dayofyear'] / 365.25)
data['weekofyear'] = data['Date'].dt.isocalendar().week.astype(int)
data

,Date,Revenue,COGS,num_active_promos,num_promo_fixed,num_promo_percentage,avg_discount_value,has_stackable,num_specific_category_promos,channel_all_channels,...,COGS_roll_std_7,Revenue_roll_mean_14,Revenue_roll_std_14,COGS_roll_mean_14,COGS_roll_std_14,Revenue_roll_mean_30,Revenue_roll_std_30,COGS_roll_mean_30,COGS_roll_std_30,Gross_Margin_lag_1
0,2014-07-04,1539342.27,1546984.48,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,1.825281e+06,7.445334e+06,1.644333e+06,6.802910e+06,1.856021e+06,6.783836e+06,1.488260e+06,5.827972e+06,1.679668e+06,0.048312
1,2014-07-05,2247607.00,2088267.20,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,3.005536e+06,7.146219e+06,2.250216e+06,6.581276e+06,2.271631e+06,6.621048e+06,1.769591e+06,5.705735e+06,1.850620e+06,-0.004965
2,2014-07-06,2992177.47,2950653.66,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,3.608661e+06,6.911414e+06,2.578808e+06,6.416333e+06,2.513368e+06,6.539260e+06,1.912327e+06,5.647226e+06,1.937245e+06,0.070893
3,2014-07-07,3321041.63,3237780.91,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,3.360326e+06,6.634352e+06,2.783707e+06,6.237848e+06,2.671089e+06,6.490474e+06,1.984615e+06,5.629741e+06,1.959761e+06,0.013877
4,2014-07-08,4389991.98,4344681.53,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,3.128331e+06,6.334362e+06,2.904499e+06,6.032275e+06,2.789336e+06,6.424784e+06,2.056957e+06,5.599474e+06,1.990244e+06,0.025071
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3098,2022-12-27,2100553.66,2184872.24,1.0,0.0,1.0,20.0,0.0,0.0,1.0,...,3.592945e+05,1.699922e+06,3.454986e+05,1.727411e+06,3.540577e+05,1.589132e+06,5.656207e+05,1.626506e+06,5.938495e+05,-0.023751
3099,2022-12-28,3448729.20,3513621.00,1.0,0.0,1.0,20.0,0.0,0.0,1.0,...,3.838777e+05,1.731940e+06,3.611613e+05,1.762933e+06,3.741305e+05,1.554530e+06,4.948911e+05,1.589892e+06,5.170779e+05,-0.040141
3100,2022-12-29,3083944.33,3170787.10,1.0,0.0,1.0,20.0,0.0,0.0,1.0,...,7.856147e+05,1.859546e+06,5.824504e+05,1.893320e+06,5.974926e+05,1.599323e+06,5.967576e+05,1.635268e+06,6.180152e+05,-0.018816
3101,2022-12-30,2884668.76,3022292.15,1.0,0.0,1.0,20.0,0.0,0.0,1.0,...,9.005524e+05,1.973090e+06,6.560662e+05,2.008776e+06,6.777455e+05,1.616831e+06,6.325077e+05,1.651148e+06,6.513871e+05,-0.028160


In [75]:
lags = [1, 2, 3, 7, 14, 30, 60, 90, 365]
windows = [7, 14, 30]

for lag in lags:
    data[f'Revenue_lag_{lag}'] = data['Revenue'].shift(lag)
    data[f'COGS_lag_{lag}'] = data['COGS'].shift(lag)

for w in windows:
    data[f'Revenue_roll_mean_{w}'] = data['Revenue'].shift(1).rolling(w).mean()
    data[f'Revenue_roll_std_{w}'] = data['Revenue'].shift(1).rolling(w).std()
    data[f'COGS_roll_mean_{w}'] = data['COGS'].shift(1).rolling(w).mean()
    data[f'COGS_roll_std_{w}'] = data['COGS'].shift(1).rolling(w).std()
data['Gross_Margin_lag_1'] = (data['Revenue'].shift(1) - data['COGS'].shift(1)) / (data['Revenue'].shift(1) + 1)
data = data.dropna().reset_index(drop=True)
data

,Date,Revenue,COGS,num_active_promos,num_promo_fixed,num_promo_percentage,avg_discount_value,has_stackable,num_specific_category_promos,channel_all_channels,...,COGS_roll_std_7,Revenue_roll_mean_14,Revenue_roll_std_14,COGS_roll_mean_14,COGS_roll_std_14,Revenue_roll_mean_30,Revenue_roll_std_30,COGS_roll_mean_30,COGS_roll_std_30,Gross_Margin_lag_1
0,2015-07-04,1184319.43,1068025.58,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,1.977082e+06,7.581862e+06,1.772348e+06,6.844267e+06,1.958725e+06,6.798247e+06,1.686370e+06,5.769029e+06,1.820623e+06,0.029889
1,2015-07-05,2966146.59,2810871.75,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,3.261735e+06,7.258626e+06,2.430517e+06,6.606900e+06,2.424781e+06,6.691412e+06,1.928395e+06,5.684228e+06,1.977081e+06,0.098195
2,2015-07-06,2490899.57,2437063.72,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,3.699248e+06,7.059391e+06,2.666119e+06,6.482904e+06,2.577789e+06,6.633970e+06,2.013859e+06,5.655704e+06,2.013040e+06,0.052349
3,2015-07-07,4185006.32,3947428.34,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,3.718545e+06,6.659375e+06,2.908515e+06,6.193689e+06,2.795363e+06,6.566849e+06,2.118137e+06,5.614168e+06,2.067268e+06,0.021613
4,2015-07-08,4333446.86,4256271.94,1.0,0.0,1.0,18.0,0.0,0.0,0.0,...,3.051328e+06,6.385831e+06,2.951026e+06,6.007810e+06,2.855735e+06,6.548433e+06,2.136909e+06,5.624522e+06,2.057779e+06,0.056769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2733,2022-12-27,2100553.66,2184872.24,1.0,0.0,1.0,20.0,0.0,0.0,1.0,...,3.592945e+05,1.699922e+06,3.454986e+05,1.727411e+06,3.540577e+05,1.589132e+06,5.656207e+05,1.626506e+06,5.938495e+05,-0.023751
2734,2022-12-28,3448729.20,3513621.00,1.0,0.0,1.0,20.0,0.0,0.0,1.0,...,3.838777e+05,1.731940e+06,3.611613e+05,1.762933e+06,3.741305e+05,1.554530e+06,4.948911e+05,1.589892e+06,5.170779e+05,-0.040141
2735,2022-12-29,3083944.33,3170787.10,1.0,0.0,1.0,20.0,0.0,0.0,1.0,...,7.856147e+05,1.859546e+06,5.824504e+05,1.893320e+06,5.974926e+05,1.599323e+06,5.967576e+05,1.635268e+06,6.180152e+05,-0.018816
2736,2022-12-30,2884668.76,3022292.15,1.0,0.0,1.0,20.0,0.0,0.0,1.0,...,9.005524e+05,1.973090e+06,6.560662e+05,2.008776e+06,6.777455e+05,1.616831e+06,6.325077e+05,1.651148e+06,6.513871e+05,-0.028160


In [76]:
current_features = [
    'year', 'month', 'day', 'dayofweek', 'is_weekend', 'is_holiday',
    'dayofyear', 'dayofyearsin', 'dayofyearcos', 'weekofyear',
    'num_active_promos', 'num_promo_percentage', 'num_promo_fixed',
    'avg_discount_value', 'has_stackable', 'num_specific_category_promos',
    'min_order_value_mean',
    'channel_all_channels', 'channel_email', 'channel_in_store', 'channel_online', 'channel_social_media'
]
lag_features_cogs = [f'COGS_lag_{lag}' for lag in lags]
lag_features_revenue = [f'Revenue_lag_{lag}' for lag in lags]
roll_features_cogs = [f'COGS_roll_mean_{w}' for w in windows] + [f'COGS_roll_std_{w}' for w in windows]
roll_features_revenue = [f'Revenue_roll_mean_{w}' for w in windows] + [f'Revenue_roll_std_{w}' for w in windows]
features_for_cogs = current_features + lag_features_cogs + roll_features_cogs
features_for_revenue = current_features + lag_features_revenue + lag_features_cogs + roll_features_revenue + roll_features_cogs + ['Gross_Margin_lag_1']
print(f"Features COGS: {len(features_for_cogs)}")
print(f"Features Revenue: {len(features_for_revenue)}")

Features COGS: 37
Features Revenue: 53


In [77]:
train_mask = data['Date'] < '2022-01-01'
val_mask = (data['Date'] >= '2022-01-01') & (data['Date'] <= '2022-12-31')

X_train_revenue = data.loc[train_mask, features_for_revenue]
y_train_revenue = data.loc[train_mask, 'Revenue']
X_val_revenue = data.loc[val_mask, features_for_revenue]
y_val_revenue = data.loc[val_mask, 'Revenue']

X_train_cogs = data.loc[train_mask, features_for_cogs]
y_train_cogs = data.loc[train_mask, 'COGS']
X_val_cogs = data.loc[val_mask, features_for_cogs]
y_val_cogs = data.loc[val_mask, 'COGS']

print(f"\nTrain: {len(X_train_revenue)} ngày, Validation: {len(X_val_revenue)} ngày")


Train: 2373 ngày, Validation: 365 ngày


In [78]:
model_lgb_rev = LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=64,subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
model_xgb_rev = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6,subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist", verbosity=0)
model_cat_rev = CatBoostRegressor(loss_function='RMSE', iterations=1000, learning_rate=0.1, depth=6, random_seed=42, verbose=0)

model_lgb_rev.fit(X_train_revenue, y_train_revenue)
model_xgb_rev.fit(X_train_revenue, y_train_revenue)
model_cat_rev.fit(X_train_revenue, y_train_revenue)

oof_lgb_rev = model_lgb_rev.predict(X_val_revenue)
oof_xgb_rev = model_xgb_rev.predict(X_val_revenue)
oof_cat_rev = model_cat_rev.predict(X_val_revenue)

oof_blend_rev = 0.4 * oof_lgb_rev + 0.35 * oof_cat_rev + 0.25 * oof_xgb_rev

print(f"LightGBM - MAE: {mean_absolute_error(y_val_revenue, oof_lgb_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, oof_lgb_rev)):,.0f}, R²: {r2_score(y_val_revenue, oof_lgb_rev):.4f}")
print(f"XGBoost - MAE: {mean_absolute_error(y_val_revenue, oof_xgb_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, oof_xgb_rev)):,.0f}, R²: {r2_score(y_val_revenue, oof_xgb_rev):.4f}")
print(f"CatBoost - MAE: {mean_absolute_error(y_val_revenue, oof_cat_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, oof_cat_rev)):,.0f}, R²: {r2_score(y_val_revenue, oof_cat_rev):.4f}")
print(f"Blend - MAE: {mean_absolute_error(y_val_revenue, oof_blend_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, oof_blend_rev)):,.0f}, R²: {r2_score(y_val_revenue, oof_blend_rev):.4f}")

LightGBM - MAE: 595,722, RMSE: 822,523, R²: 0.7585
XGBoost - MAE: 569,939, RMSE: 789,279, R²: 0.7776
CatBoost - MAE: 559,323, RMSE: 772,024, R²: 0.7873
Blend - MAE: 567,059, RMSE: 783,794, R²: 0.7807


In [79]:
stack_train_rev = np.vstack([oof_lgb_rev, oof_cat_rev, oof_xgb_rev]).T
lvl2_rev = Ridge(alpha=1.0)
lvl2_rev.fit(stack_train_rev, y_val_revenue)
pred_stack_rev = lvl2_rev.predict(stack_train_rev)

print(f"Stacked Ridge - MAE: {mean_absolute_error(y_val_revenue, pred_stack_rev):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_revenue, pred_stack_rev)):,.0f}, R²: {r2_score(y_val_revenue, pred_stack_rev):.6f}")
print(f"Ridge coefs: {lvl2_rev.coef_}")

Stacked Ridge - MAE: 554,662, RMSE: 758,933, R²: 0.794414
Ridge coefs: [-0.29494753  0.63918885  0.61326039]


In [80]:
model_lgb_cogs = LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=64,subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
model_xgb_cogs = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6,subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist", verbosity=0)
model_cat_cogs = CatBoostRegressor(loss_function='RMSE', iterations=1000, learning_rate=0.1,depth=6, random_seed=42, verbose=0)

model_lgb_cogs.fit(X_train_cogs, y_train_cogs)
model_xgb_cogs.fit(X_train_cogs, y_train_cogs)
model_cat_cogs.fit(X_train_cogs, y_train_cogs)

oof_lgb_cogs = model_lgb_cogs.predict(X_val_cogs)
oof_xgb_cogs = model_xgb_cogs.predict(X_val_cogs)
oof_cat_cogs = model_cat_cogs.predict(X_val_cogs)

oof_blend_cogs = 0.4 * oof_lgb_cogs + 0.35 * oof_cat_cogs + 0.25 * oof_xgb_cogs

print(f"LightGBM - MAE: {mean_absolute_error(y_val_cogs, oof_lgb_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, oof_lgb_cogs)):,.0f}, R²: {r2_score(y_val_cogs, oof_lgb_cogs):.4f}")
print(f"XGBoost  - MAE: {mean_absolute_error(y_val_cogs, oof_xgb_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, oof_xgb_cogs)):,.0f}, R²: {r2_score(y_val_cogs, oof_xgb_cogs):.4f}")
print(f"CatBoost - MAE: {mean_absolute_error(y_val_cogs, oof_cat_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, oof_cat_cogs)):,.0f}, R²: {r2_score(y_val_cogs, oof_cat_cogs):.4f}")
print(f"Blend    - MAE: {mean_absolute_error(y_val_cogs, oof_blend_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, oof_blend_cogs)):,.0f}, R²: {r2_score(y_val_cogs, oof_blend_cogs):.4f}")

stack_train_cogs = np.vstack([oof_lgb_cogs, oof_cat_cogs, oof_xgb_cogs]).T
lvl2_cogs = Lasso(alpha=0.1, max_iter=5000)
lvl2_cogs.fit(stack_train_cogs, y_val_cogs)
pred_stack_cogs = lvl2_cogs.predict(stack_train_cogs)

print(f"Stacked Lasso - MAE: {mean_absolute_error(y_val_cogs, pred_stack_cogs):,.0f}, RMSE: {np.sqrt(mean_squared_error(y_val_cogs, pred_stack_cogs)):,.0f}, R²: {r2_score(y_val_cogs, pred_stack_cogs):.4f}")
print(f"Lasso coefs: {lvl2_cogs.coef_}")

LightGBM - MAE: 519,007, RMSE: 714,163, R²: 0.7603
XGBoost  - MAE: 499,755, RMSE: 686,519, R²: 0.7785
CatBoost - MAE: 483,528, RMSE: 649,749, R²: 0.8016
Blend    - MAE: 495,382, RMSE: 675,643, R²: 0.7854
Stacked Lasso - MAE: 479,047, RMSE: 637,982, R²: 0.8087
Lasso coefs: [-0.47989737  1.15359771  0.30774378]


In [97]:
from prophet import Prophet

prophet_train_cogs = data.loc[train_mask, ['Date', 'COGS']].rename(columns={'Date': 'ds', 'COGS': 'y'})
model_prophet_cogs = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_prophet_cogs.fit(prophet_train_cogs)

prophet_train_rev = data.loc[train_mask, ['Date', 'Revenue']].rename(columns={'Date': 'ds', 'Revenue': 'y'})
model_prophet_rev = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_prophet_rev.fit(prophet_train_rev)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
10:19:56 - cmdstanpy - INFO - Chain [1] start processing
10:19:57 - cmdstanpy - INFO - Chain [1] done processing
10:19:57 - cmdstanpy - INFO - Chain [1] start processing
10:19:58 - cmdstanpy - INFO - Chain [1] done processing


In [99]:
prophet_pred_rev_train = model_prophet_rev.predict(pd.DataFrame({'ds': data.loc[train_mask, 'Date']}))['yhat'].values

residuals_rev_train = data.loc[train_mask, 'Revenue'].values - prophet_pred_rev_train

model_lgb_rev.fit(X_train_revenue, residuals_rev_train)
model_xgb_rev.fit(X_train_revenue, residuals_rev_train)
model_cat_rev.fit(X_train_revenue, residuals_rev_train)

oof_lgb_rev_res = model_lgb_rev.predict(X_val_revenue)
oof_xgb_rev_res = model_xgb_rev.predict(X_val_revenue)
oof_cat_rev_res = model_cat_rev.predict(X_val_revenue)

stack_train_rev_res = np.vstack([oof_lgb_rev_res, oof_cat_rev_res, oof_xgb_rev_res]).T
y_val_rev_res = data.loc[val_mask, 'Revenue'].values - model_prophet_rev.predict(pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values

lvl2_rev = Lasso(alpha=0.1, max_iter=5000)
lvl2_rev.fit(stack_train_rev_res, y_val_rev_res)
print(f"Revenue Lasso coefs: {lvl2_rev.coef_}")

pred_stack_rev_res = lvl2_rev.predict(stack_train_rev_res)
pred_rev_final = model_prophet_rev.predict(pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values + pred_stack_rev_res
print(f"Revenue Final (Prophet + Stacked Residuals) - R²: {r2_score(y_val_revenue, pred_rev_final):.4f}")

Revenue Lasso coefs: [0.22974422 0.18914098 0.64739338]
Revenue Final (Prophet + Stacked Residuals) - R²: 0.7812


In [98]:

prophet_pred_cogs_train = model_prophet_cogs.predict(pd.DataFrame({'ds': data.loc[train_mask, 'Date']}))['yhat'].values

residuals_cogs_train = data.loc[train_mask, 'COGS'].values - prophet_pred_cogs_train

model_lgb_cogs.fit(X_train_cogs, residuals_cogs_train)
model_xgb_cogs.fit(X_train_cogs, residuals_cogs_train)
model_cat_cogs.fit(X_train_cogs, residuals_cogs_train)

oof_lgb_cogs_res = model_lgb_cogs.predict(X_val_cogs)
oof_xgb_cogs_res = model_xgb_cogs.predict(X_val_cogs)
oof_cat_cogs_res = model_cat_cogs.predict(X_val_cogs)

stack_train_cogs_res = np.vstack([oof_lgb_cogs_res, oof_cat_cogs_res, oof_xgb_cogs_res]).T
y_val_cogs_res = data.loc[val_mask, 'COGS'].values - model_prophet_cogs.predict(pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values

lvl2_cogs = Lasso(alpha=0.1, max_iter=5000)
lvl2_cogs.fit(stack_train_cogs_res, y_val_cogs_res)
print(f"COGS Lasso coefs: {lvl2_cogs.coef_}")

pred_stack_cogs_res = lvl2_cogs.predict(stack_train_cogs_res)
pred_cogs_final = model_prophet_cogs.predict(pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values + pred_stack_cogs_res
print(f"COGS Final (Prophet + Stacked Residuals) - R²: {r2_score(y_val_cogs, pred_cogs_final):.4f}")

COGS Lasso coefs: [-0.59619652  0.02478231  1.58839457]
COGS Final (Prophet + Stacked Residuals) - R²: 0.7753


In [ ]:
def recursive_forecast_with_prophet(historical_data, forecast_dates,
                                    model_prophet_cogs, model_prophet_rev,
                                    model_lgb_c, model_xgb_c, model_cat_c, lvl2_c,
                                    model_lgb_r, model_xgb_r, model_cat_r, lvl2_r,
                                    features_for_cogs, features_for_revenue,
                                    lags, windows, all_promo_features, holidays_and_sales):
    """
    Recursive forecast kết hợp Prophet + Stacking
    """
    historical = historical_data.copy()
    predictions = []
    
    for forecast_date in forecast_dates:
        features = {}

        future_df = pd.DataFrame({'ds': [forecast_date]})
        prophet_cogs_pred = model_prophet_cogs.predict(future_df)['yhat'].values[0]
        prophet_rev_pred = model_prophet_rev.predict(future_df)['yhat'].values[0]

        day_info = all_promo_features[all_promo_features['Date'] == forecast_date]
        
        features['year'] = forecast_date.year
        features['month'] = forecast_date.month
        features['day'] = forecast_date.day
        features['dayofweek'] = forecast_date.dayofweek
        features['is_weekend'] = 1 if forecast_date.dayofweek in [5, 6] else 0
        features['dayofyear'] = forecast_date.dayofyear
        features['dayofyearsin'] = np.sin(2 * np.pi * forecast_date.dayofyear / 365.25)
        features['dayofyearcos'] = np.cos(2 * np.pi * forecast_date.dayofyear / 365.25)
        features['weekofyear'] = forecast_date.isocalendar().week
        features['is_holiday'] = 1 if forecast_date.strftime('%m-%d') in holidays_and_sales else 0
        
        promo_cols = ['num_active_promos', 'num_promo_percentage', 'num_promo_fixed','avg_discount_value', 'has_stackable', 'num_specific_category_promos','min_order_value_mean', 'channel_all_channels', 'channel_email','channel_in_store', 'channel_online', 'channel_social_media']
        for col in promo_cols:
            if len(day_info) > 0 and col in day_info.columns:
                features[col] = day_info[col].values[0]
            else:
                features[col] = 0

        for lag in lags:
            idx = len(historical) - lag
            features[f'COGS_lag_{lag}'] = historical['COGS'].iloc[idx] if idx >= 0 else historical['COGS'].mean()
        
        for w in windows:
            if len(historical) >= w:
                features[f'COGS_roll_mean_{w}'] = historical['COGS'].iloc[-w:].mean()
                features[f'COGS_roll_std_{w}'] = historical['COGS'].iloc[-w:].std()
            else:
                features[f'COGS_roll_mean_{w}'] = historical['COGS'].mean()
                features[f'COGS_roll_std_{w}'] = 0
        
        X_cogs = pd.DataFrame([features]).reindex(columns=features_for_cogs, fill_value=0)
        
        p_lgb_c = model_lgb_c.predict(X_cogs)[0]
        p_xgb_c = model_xgb_c.predict(X_cogs)[0]
        p_cat_c = model_cat_c.predict(X_cogs)[0]
        cogs_residual = lvl2_c.predict(np.array([[p_lgb_c, p_cat_c, p_xgb_c]]))[0]

        pred_cogs = prophet_cogs_pred + cogs_residual
        pred_cogs = max(0, pred_cogs)

        for lag in lags:
            idx = len(historical) - lag
            features[f'Revenue_lag_{lag}'] = historical['Revenue'].iloc[idx] if idx >= 0 else historical['Revenue'].mean()
        
        for w in windows:
            if len(historical) >= w:
                features[f'Revenue_roll_mean_{w}'] = historical['Revenue'].iloc[-w:].mean()
                features[f'Revenue_roll_std_{w}'] = historical['Revenue'].iloc[-w:].std()
            else:
                features[f'Revenue_roll_mean_{w}'] = historical['Revenue'].mean()
                features[f'Revenue_roll_std_{w}'] = 0
        
        if len(historical) >= 1:
            rev_l1 = historical['Revenue'].iloc[-1]
            cogs_l1 = historical['COGS'].iloc[-1]
            features['Gross_Margin_lag_1'] = (rev_l1 - cogs_l1) / (rev_l1 + 1)
        else:
            features['Gross_Margin_lag_1'] = 0
        
        X_rev = pd.DataFrame([features]).reindex(columns=features_for_revenue, fill_value=0)
        
        p_lgb_r = model_lgb_r.predict(X_rev)[0]
        p_xgb_r = model_xgb_r.predict(X_rev)[0]
        p_cat_r = model_cat_r.predict(X_rev)[0]
        rev_residual = lvl2_r.predict(np.array([[p_lgb_r, p_cat_r, p_xgb_r]]))[0]

        pred_rev = prophet_rev_pred + rev_residual
        pred_rev = max(0, pred_rev)

        predictions.append({
            'Date': forecast_date,
            'COGS': pred_cogs,
            'Revenue': pred_rev
        })

        new_row = historical.iloc[-1].copy()
        new_row['Date'] = forecast_date
        new_row['COGS'] = pred_cogs
        new_row['Revenue'] = pred_rev
        historical = pd.concat([historical, pd.DataFrame([new_row])], ignore_index=True)
    
    return pd.DataFrame(predictions)

In [100]:
val_dates = pd.date_range('2022-01-01', '2022-12-31', freq='D')

recursive_val = recursive_forecast_with_prophet(
    data[data['Date'] < '2022-01-01'],
    val_dates,
    model_prophet_cogs, model_prophet_rev,
    model_lgb_cogs, model_xgb_cogs, model_cat_cogs, lvl2_cogs,
    model_lgb_rev, model_xgb_rev, model_cat_rev, lvl2_rev,
    features_for_cogs, features_for_revenue,
    lags, windows, all_promo_features, holidays_and_sales
)

comparison = recursive_val.merge(
    data.loc[val_mask, ['Date', 'Revenue', 'COGS']], 
    on='Date', 
    suffixes=('_pred', '_actual')
)

print(f"Revenue:")
print(f"  RMSE: {np.sqrt(mean_squared_error(comparison['Revenue_actual'], comparison['Revenue_pred'])):,.0f}")
print(f"  MAE:  {mean_absolute_error(comparison['Revenue_actual'], comparison['Revenue_pred']):,.0f}")
print(f"  R²:   {r2_score(comparison['Revenue_actual'], comparison['Revenue_pred']):.4f}")

print(f"\nCOGS:")
print(f"  RMSE: {np.sqrt(mean_squared_error(comparison['COGS_actual'], comparison['COGS_pred'])):,.0f}")
print(f"  MAE:  {mean_absolute_error(comparison['COGS_actual'], comparison['COGS_pred']):,.0f}")
print(f"  R²:   {r2_score(comparison['COGS_actual'], comparison['COGS_pred']):.4f}")

Revenue:
  RMSE: 918,135
  MAE:  660,214
  R²:   0.6991

COGS:
  RMSE: 739,566
  MAE:  539,385
  R²:   0.7429


In [101]:
model_lgb_cogs_final = LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=64,subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
model_xgb_cogs_final = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6,subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist", verbosity=0)
model_cat_cogs_final = CatBoostRegressor(loss_function='RMSE', iterations=1000, learning_rate=0.1,depth=6, random_seed=42, verbose=0)

model_lgb_cogs_final.fit(data[features_for_cogs], data['COGS'])
model_xgb_cogs_final.fit(data[features_for_cogs], data['COGS'])
model_cat_cogs_final.fit(data[features_for_cogs], data['COGS'])

oof_cogs_full_lgb = model_lgb_cogs_final.predict(data[features_for_cogs])
oof_cogs_full_xgb = model_xgb_cogs_final.predict(data[features_for_cogs])
oof_cogs_full_cat = model_cat_cogs_final.predict(data[features_for_cogs])
stack_cogs_full = np.vstack([oof_cogs_full_lgb, oof_cogs_full_cat, oof_cogs_full_xgb]).T
lvl2_cogs_final = Lasso(alpha=0.1, max_iter=5000)
lvl2_cogs_final.fit(stack_cogs_full, data['COGS'])

model_lgb_rev_final = LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=64,subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
model_xgb_rev_final = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6,subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist", verbosity=0)
model_cat_rev_final = CatBoostRegressor(loss_function='RMSE', iterations=1000, learning_rate=0.1,depth=6, random_seed=42, verbose=0)

model_lgb_rev_final.fit(data[features_for_revenue], data['Revenue'])
model_xgb_rev_final.fit(data[features_for_revenue], data['Revenue'])
model_cat_rev_final.fit(data[features_for_revenue], data['Revenue'])

oof_rev_full_lgb = model_lgb_rev_final.predict(data[features_for_revenue])
oof_rev_full_xgb = model_xgb_rev_final.predict(data[features_for_revenue])
oof_rev_full_cat = model_cat_rev_final.predict(data[features_for_revenue])
stack_rev_full = np.vstack([oof_rev_full_lgb, oof_rev_full_cat, oof_rev_full_xgb]).T
lvl2_rev_final = Lasso(alpha=0.1, max_iter=5000)
lvl2_rev_final.fit(stack_rev_full, data['Revenue'])

print(f"Lasso COGS coefs: {lvl2_cogs_final.coef_}")
print(f"Lasso Revenue coefs: {lvl2_rev_final.coef_}")

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.824e+12, tolerance: 1.348e+12
  model = cd_fast.enet_coordinate_descent(


Lasso COGS coefs: [ 0.44126635 -0.34476404  0.9052159 ]
Lasso Revenue coefs: [ 0.43625924 -0.24039722  0.80574649]


In [91]:
test_dates = pd.date_range('2023-01-01', '2024-07-01', freq='D')
submission = recursive_forecast_with_stacking(
    data, test_dates,
    model_lgb_cogs_final, model_xgb_cogs_final, model_cat_cogs_final, lvl2_cogs_final,
    model_lgb_rev_final, model_xgb_rev_final, model_cat_rev_final, lvl2_rev_final,
    features_for_cogs, features_for_revenue,
    lags, windows, all_promo_features, holidays_and_sales
)

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))


Submission shape: (548, 3)
        Date          COGS       Revenue
0 2023-01-01  1.887236e+06  1.805697e+06
1 2023-01-02  1.659813e+06  1.355346e+06
2 2023-01-03  9.178744e+05  9.452796e+05
3 2023-01-04  7.923953e+05  9.630737e+05
4 2023-01-05  1.003082e+06  1.102022e+06
5 2023-01-06  1.036826e+06  1.154945e+06
6 2023-01-07  1.066686e+06  1.316220e+06
7 2023-01-08  1.370254e+06  1.658586e+06
8 2023-01-09  1.481954e+06  1.842226e+06
9 2023-01-10  1.532057e+06  1.767218e+06


In [92]:
submission[['Date', 'Revenue', 'COGS']].to_csv('submission.csv', index=False)